# VQE for Hydrogen (H2)

Find the ground-state energy of the H2 molecule by minimizing an expectation value — the headline of this section.

**Objectives:**
- State the variational principle and why it turns ground-state finding into optimization
- Run a symmetry-tapered SINGLE-QUBIT VQE whose energy has the closed form `E(theta) = c0 + cz*cos(theta) + cx*sin(theta)`
- Run the full FOUR-QUBIT VQE with the kit's `h2_double_excitation(theta)` Givens ansatz
- Confirm both land on the exact FCI energy and verify each against exact diagonalization of the Hamiltonian matrix

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Use local simulator by default (free, instant)
device = LocalSimulator()

In [ ]:
# --- Chemistry kit (auto-provided; real STO-3G H2 data + exact helpers) ---
# H2 Jordan-Wigner qubit Hamiltonian at R = 0.75 Angstrom (STO-3G, big-endian:
# character k of each Pauli string acts on qubit k, qubit 0 = leftmost = MSB).
# Coefficients are REAL precomputed values (PennyLane differentiable Hartree-Fock).
H2_TERMS = [
    ("IIII", -0.1097305),
    ("IIIZ", -0.21886309),
    ("IIZI", -0.21886309),
    ("IIZZ", 0.17395379),
    ("IZII", 0.16988453),
    ("IZIZ", 0.12005143),
    ("IZZI", 0.16549432),
    ("XXYY", -0.04544288),
    ("XYYX", 0.04544288),
    ("YXXY", 0.04544288),
    ("YYXX", -0.04544288),
    ("ZIII", 0.16988453),
    ("ZIIZ", 0.16549432),
    ("ZIZI", 0.12005143),
    ("ZZII", 0.16821199),
]
H2_FCI = -1.137117   # exact ground-state energy (lowest eigenvalue), Hartree
H2_HF = -1.116151    # Hartree-Fock energy <1100|H|1100>, Hartree
# Z2 symmetry-tapered single-qubit form: H = c0*I + cz*Z + cx*X (ground = c0 - hypot(cz, cx))
H2_C0, H2_CZ, H2_CX = -0.338656, 0.777495, 0.181772

_PAULI = {
    "I": np.array([[1, 0], [0, 1]], dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def pauli_matrix(pauli_string):
    """Dense matrix of a big-endian Pauli string (qubit 0 = leftmost factor)."""
    m = np.array([[1.0 + 0j]])
    for ch in pauli_string:
        m = np.kron(m, _PAULI[ch])
    return m

def hamiltonian_matrix(terms):
    """Dense Hamiltonian sum(coeff * Pauli) from a list of (pauli_string, coeff)."""
    n = len(terms[0][0])
    h = np.zeros((2 ** n, 2 ** n), dtype=complex)
    for s, c in terms:
        h += c * pauli_matrix(s)
    return h

def hamiltonian_energy(state_vector, terms):
    """Expectation <psi|H|psi> for a qcsim state vector (real, in Hartree)."""
    h = hamiltonian_matrix(terms)
    return float(np.real(np.vdot(state_vector, h @ state_vector)))

def h2_double_excitation(theta):
    """HF |1100> plus the single Givens double excitation |1100> <-> |0011>.
    At the optimal theta this ansatz reaches the exact H2 ground state."""
    c = Circuit()
    c.x(0); c.x(1)
    c.cnot(2, 3); c.cnot(0, 2); c.h(0); c.h(3); c.cnot(0, 1); c.cnot(2, 3)
    c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(0, 3); c.h(3); c.cnot(3, 1); c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(2, 1); c.cnot(2, 0); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(3, 1); c.h(3); c.cnot(0, 3); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(0, 1); c.cnot(2, 0); c.h(0); c.h(3); c.cnot(0, 2); c.cnot(2, 3)
    return c
# --- end chemistry kit ---

## 1. The molecule, already an operator

The previous notebooks did the hard physics: two hydrogen atoms in the STO-3G basis, mapped through Jordan-Wigner, become a four-qubit operator that is a weighted sum of fifteen Pauli strings. The chemistry kit hands us that operator directly as `H2_TERMS` — a list of `(pauli_string, coeff)` pairs, big-endian so character `k` of each string acts on qubit `k`.

Finding the molecule's ground-state energy means finding the **lowest eigenvalue** of that operator. The kit also gives us the answers to check against:

- `H2_FCI` — the exact ground energy (full configuration interaction), the number we are chasing.
- `H2_HF` — the Hartree-Fock energy of the mean-field state `|1100>`, an upper bound that VQE should beat.

Let us look at the operator and the targets.

In [ ]:
print(f"H2_TERMS: {len(H2_TERMS)} Pauli terms")
for pauli, coeff in H2_TERMS[:5]:
    print(f"  {coeff:+.6f} * {pauli}")
print("  ...")

print(f"\nHartree-Fock energy  H2_HF  = {H2_HF:.6f} Ha   (mean-field |1100>)")
print(f"Exact ground energy  H2_FCI = {H2_FCI:.6f} Ha   (target)")
print(f"Correlation energy   HF - FCI = {H2_HF - H2_FCI:.6f} Ha   (what VQE must recover)")

## 2. The variational principle

VQE rests on one fact from physics. For **any** trial state `|psi(theta)>` we can prepare,

$$\langle \psi(\theta)|H|\psi(\theta)\rangle \ge E_{\text{ground}}.$$

The expected energy of any state is an upper bound on the true ground energy — you can never measure below the floor, only approach it. That turns ground-state finding into **minimization**: prepare a parameterized state, measure `<H>`, hand the number to a classical optimizer, adjust `theta`, repeat.

We compute `<psi|H|psi>` exactly with the kit's `hamiltonian_energy(state_vector, terms)`, which contracts the dense Hamiltonian against a circuit's `state_vector()`. First let us confirm the floor itself: the smallest eigenvalue of `hamiltonian_matrix(H2_TERMS)` must equal `H2_FCI`.

In [ ]:
H = hamiltonian_matrix(H2_TERMS)          # dense 16x16 Hermitian matrix
eigvals = np.linalg.eigvalsh(H)            # ascending real eigenvalues
E_exact = eigvals[0]

print(f"Hamiltonian matrix shape : {H.shape}")
print(f"Lowest eigenvalue        : {E_exact:.6f} Ha")
print(f"H2_FCI                   : {H2_FCI:.6f} Ha")

# Exact diagonalization must reproduce the kit's FCI value.
assert np.isclose(E_exact, H2_FCI, atol=1e-6), (E_exact, H2_FCI)
print("\nOK: exact diagonalization reproduces H2_FCI.")

## 3. Ansatz A — the symmetry-tapered single qubit

H2 carries enough symmetry that, after tapering away the conserved quantities, the entire four-qubit problem collapses onto a **single qubit** with a three-term Hamiltonian:

$$H_{\text{tap}} = c_0\, I + c_z\, Z + c_x\, X,$$

with coefficients the kit exposes as `H2_C0`, `H2_CZ`, `H2_CX`. The ground energy of this operator is `c0 - hypot(cz, cx)`, which equals `H2_FCI` exactly.

The ansatz is a single rotation, `Ry(theta)|0>`. That state is `cos(theta/2)|0> + sin(theta/2)|1>`, so its expectations are

$$\langle Z\rangle = \cos\theta, \qquad \langle X\rangle = \sin\theta,$$

and the energy takes the clean closed form

$$E(\theta) = c_0 + c_z\cos\theta + c_x\sin\theta.$$

Because one qubit's `Ry` can reach **every** state of that qubit, this VQE is not approximate — it lands exactly on `H2_FCI`. Let us first confirm the closed form agrees with what the simulator actually measures from `state_vector()`, then optimize.

In [ ]:
def tapered_state_expectations(theta):
    """<Z> and <X> of Ry(theta)|0>, read from the qcsim state vector."""
    circ = Circuit().ry(0, theta)
    psi = circ.state_vector()          # length-2 vector [amp|0>, amp|1>]
    a0, a1 = psi[0], psi[1]
    # <Z> = |a0|^2 - |a1|^2 ;  <X> = 2 Re(conj(a0) a1)
    exp_z = (np.abs(a0) ** 2 - np.abs(a1) ** 2).real
    exp_x = (2 * np.conj(a0) * a1).real
    return exp_z, exp_x


def tapered_energy(theta):
    """E(theta) = c0 + cz*<Z> + cx*<X> for the tapered single-qubit H2."""
    exp_z, exp_x = tapered_state_expectations(theta)
    return H2_C0 + H2_CZ * exp_z + H2_CX * exp_x


# The measured expectations must match the cos/sin closed form.
for t in [0.0, 0.7, 1.9, 3.1]:
    ez, ex = tapered_state_expectations(t)
    assert np.isclose(ez, np.cos(t), atol=1e-9)
    assert np.isclose(ex, np.sin(t), atol=1e-9)
print("OK: <Z> = cos(theta) and <X> = sin(theta) from state_vector().")
print(f"Tapered coefficients: c0={H2_C0:+.6f}  cz={H2_CZ:+.6f}  cx={H2_CX:+.6f}")

### Optimize with a numpy grid, then refine

No scipy: a coarse grid over `theta in [0, 2pi)` locates the basin, then a fine grid around the best point refines it. The minimum of `E(theta)` must reach `H2_FCI` (atol 1e-4), and the variational principle says every sampled energy must sit **at or above** the floor.

In [ ]:
# Coarse grid search.
coarse = np.linspace(0.0, 2 * np.pi, 400, endpoint=False)
E_coarse = np.array([tapered_energy(t) for t in coarse])
i_best = int(np.argmin(E_coarse))
t_best = coarse[i_best]

# Local refinement on a fine grid around the coarse minimum.
step = coarse[1] - coarse[0]
fine = np.linspace(t_best - step, t_best + step, 400)
E_fine = np.array([tapered_energy(t) for t in fine])
j_best = int(np.argmin(E_fine))
theta_opt = fine[j_best]
E_tapered_min = E_fine[j_best]

print(f"Optimal theta : {theta_opt:.6f} rad")
print(f"VQE energy    : {E_tapered_min:.6f} Ha")
print(f"H2_FCI        : {H2_FCI:.6f} Ha")
print(f"Error         : {abs(E_tapered_min - H2_FCI):.2e} Ha")

# Lands exactly on FCI (single qubit reaches every state).
assert np.isclose(E_tapered_min, H2_FCI, atol=1e-4), E_tapered_min
# Variational floor: no sampled energy dips below the ground energy.
assert np.all(E_coarse >= H2_FCI - 1e-9)
assert np.all(E_fine >= H2_FCI - 1e-9)
print("\nOK: tapered VQE reaches H2_FCI and never breaks the variational floor.")

### The one-qubit energy landscape

The whole problem fits in a single picture: a sinusoid `E(theta) = c0 + cz*cos(theta) + cx*sin(theta)`, with the variational floor `H2_FCI` exactly tangent to its minimum and the Hartree-Fock value `H2_HF` floating above.

In [ ]:
theta_grid = np.linspace(0.0, 2 * np.pi, 600)
E_grid = np.array([tapered_energy(t) for t in theta_grid])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(theta_grid, E_grid, color="#2b6cb0", lw=2, label=r"$E(\theta)=c_0+c_z\cos\theta+c_x\sin\theta$")
ax.axhline(H2_FCI, color="#2f855a", ls="--", lw=1.5, label=f"H2_FCI = {H2_FCI:.4f} Ha (floor)")
ax.axhline(H2_HF, color="#c05621", ls=":", lw=1.5, label=f"H2_HF = {H2_HF:.4f} Ha")
ax.scatter([theta_opt], [E_tapered_min], color="#2f855a", zorder=5, s=70, label="VQE minimum")
ax.set_xlabel(r"$\theta$ (rad)")
ax.set_ylabel("Energy (Hartree)")
ax.set_title("Tapered single-qubit H2 energy landscape")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Ansatz B — the full four-qubit VQE

The tapered qubit is elegant but special to H2. The honest, generalizable workflow runs on the full four-qubit operator. The kit's `h2_double_excitation(theta)` builds exactly the right circuit: it prepares the Hartree-Fock state `|1100>`, then applies a single Givens **double excitation** that mixes in `|0011>` by angle `theta`. At the optimal angle this reaches the exact H2 ground state.

We evaluate the energy with `hamiltonian_energy(circuit.state_vector(), H2_TERMS)` — the same `<psi|H|psi>` contraction, now against the full four-qubit state vector. Then we minimize over a `theta` grid exactly as before.

In [ ]:
def double_excitation_energy(theta):
    """<psi|H|psi> for the kit's HF + Givens double-excitation circuit."""
    circ = h2_double_excitation(theta)
    return hamiltonian_energy(circ.state_vector(), H2_TERMS)


# Sanity: at theta = 0 the circuit is bare Hartree-Fock |1100>, energy = H2_HF.
E_at_zero = double_excitation_energy(0.0)
print(f"Energy at theta=0 (HF |1100>) : {E_at_zero:.6f} Ha  (expect H2_HF = {H2_HF:.6f})")
assert np.isclose(E_at_zero, H2_HF, atol=1e-6)

# Grid search over the single excitation angle.
theta_dbl = np.linspace(-np.pi, np.pi, 800)
E_dbl = np.array([double_excitation_energy(t) for t in theta_dbl])
k_best = int(np.argmin(E_dbl))
theta_dbl_opt = theta_dbl[k_best]
E_dbl_min = E_dbl[k_best]

print(f"\nOptimal theta : {theta_dbl_opt:.6f} rad")
print(f"VQE energy    : {E_dbl_min:.6f} Ha")
print(f"H2_FCI        : {H2_FCI:.6f} Ha")
print(f"Error         : {abs(E_dbl_min - H2_FCI):.2e} Ha")

# Four-qubit VQE reaches FCI ...
assert np.isclose(E_dbl_min, H2_FCI, atol=1e-3), E_dbl_min
# ... and respects the variational floor at every angle.
assert np.all(E_dbl >= H2_FCI - 1e-9)
print("\nOK: four-qubit VQE reaches H2_FCI and stays above the floor.")

### The four-qubit landscape

A different curve, the same physics. Sweeping the double-excitation angle traces a well whose bottom touches `H2_FCI`; Hartree-Fock sits at `theta = 0`, and the optimizer slides down into the correlated ground state.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(theta_dbl, E_dbl, color="#6b46c1", lw=2, label="E(theta), 4-qubit Givens ansatz")
ax.axhline(H2_FCI, color="#2f855a", ls="--", lw=1.5, label=f"H2_FCI = {H2_FCI:.4f} Ha (floor)")
ax.scatter([0.0], [E_at_zero], color="#c05621", zorder=5, s=60, label="Hartree-Fock (theta=0)")
ax.scatter([theta_dbl_opt], [E_dbl_min], color="#2f855a", zorder=5, s=70, label="VQE minimum")
ax.set_xlabel(r"$\theta$ (rad)")
ax.set_ylabel("Energy (Hartree)")
ax.set_title("Four-qubit H2 VQE: HF + Givens double excitation")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Three roads to one number

Exact diagonalization, the tapered single-qubit VQE, and the full four-qubit VQE all converge on the same ground energy. Two are quantum-variational minimizations; one is a direct eigenvalue solve. They agree because they are three views of the identical operator `H2_TERMS`.

In [ ]:
print(f"{'Method':<34}{'Energy (Ha)':>14}{'vs FCI':>12}")
print("-" * 60)
print(f"{'Exact diagonalization':<34}{E_exact:>14.6f}{E_exact - H2_FCI:>12.2e}")
print(f"{'VQE A: tapered 1-qubit':<34}{E_tapered_min:>14.6f}{E_tapered_min - H2_FCI:>12.2e}")
print(f"{'VQE B: 4-qubit double-excitation':<34}{E_dbl_min:>14.6f}{E_dbl_min - H2_FCI:>12.2e}")
print(f"{'Hartree-Fock (reference)':<34}{H2_HF:>14.6f}{H2_HF - H2_FCI:>12.2e}")
print("-" * 60)
print(f"{'H2_FCI (target)':<34}{H2_FCI:>14.6f}{0.0:>12.2e}")

assert np.isclose(E_tapered_min, E_dbl_min, atol=1e-3)
print("\nBoth VQE ansatze agree with exact diagonalization and with each other.")

## Exercises

```python
# 1. Coordinate-descent instead of grid search.
#    Replace the tapered grid search with a coordinate/golden-section style
#    refinement: start at theta=0, repeatedly halve a bracketing window around
#    the current best. Confirm it still reaches H2_FCI to atol 1e-4.

# 2. Parameter-shift gradient.
#    For the tapered ansatz, dE/dtheta = (E(theta+pi/2) - E(theta-pi/2)) / 2.
#    Verify this against a finite-difference derivative, then do a few steps of
#    gradient descent and watch theta converge to theta_opt.

# 3. Hartree-Fock gap.
#    The correlation energy H2_HF - H2_FCI is what VQE recovers. Compute it, and
#    confirm the four-qubit VQE at theta=0 equals H2_HF exactly (pure HF state).

# 4. Off-optimum states stay above the floor.
#    Pick 10 random thetas (use the seeded np.random) for each ansatz and assert
#    every energy is >= H2_FCI. This is the variational principle as a test.
```

## Summary

- A molecule is an operator (`H2_TERMS`), the operator is a matrix (`hamiltonian_matrix`), and its lowest eigenvalue is the ground-state energy (`H2_FCI`).
- The **variational principle** guarantees `<psi(theta)|H|psi(theta)> >= E_ground`, turning ground-state finding into minimization — confirmed here by every sampled energy staying at or above the floor.
- The symmetry-**tapered single qubit** has the exact closed form `E(theta) = c0 + cz*cos(theta) + cx*sin(theta)`; because one qubit's `Ry` reaches every state, its minimum lands exactly on `H2_FCI`.
- The full **four-qubit VQE** with `h2_double_excitation(theta)` — Hartree-Fock plus one Givens double excitation — reaches the same FCI energy by recovering the correlation energy missing from `H2_HF`.
- All three methods (exact diagonalization, both VQE ansatze) agree because they are views of one operator.

**Next:** [`04-vqe-lih.ipynb`](04-vqe-lih.ipynb) — scale VQE to LiH, where the ansatz can no longer reach every state and active-space selection starts to matter.